# EIE Agent — Framework-Driven HITL Interrupts

This notebook demonstrates **framework-driven** confirmation interrupts (like LangGraph's `interrupt()`).

The LLM just calls data tools normally. The `_check_tool_confirmation` hook in `EIEAgent` **automatically** halts execution after `set_datetime_tool`, `get_place_tool`, and `collections_rag_tool` — no `ask_human` tool needed, no prompt engineering for confirmations.

**Flow:** LLM calls MCP tool → tool result returns → framework hook fires → `stream.cancel()` → `HumanInputRequiredEvent` → user responds → agent resumes.

In [ ]:
import os

from akd._base import (
    RunContext,
    ThinkingEvent,
    StreamingTokenEvent,
    CompletedEvent,
    ToolCallingEvent,
    ToolResultEvent,
    HumanInputRequiredEvent,
    HumanResponseEvent,
)
from akd._base.structures import HumanResponse

from akd_ext.agents import EIEAgent, EIEAgentInputSchema
from akd_ext.agents.eie_agent import EIEAgentConfig

os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "sk-...")
os.environ["EIE_MCP_KEY"] = os.environ.get("EIE_MCP_KEY", "fmcp_...")

## Create Agent

No custom prompt or tools needed. `EIEAgent` has `_check_tool_confirmation` built in — it auto-interrupts after `set_datetime_tool`, `get_place_tool`, and `collections_rag_tool`.

In [ ]:
agent = EIEAgent()

print(f"Model: {agent.config.model_name}")
print(f"Confirmation tools: {agent.CONFIRMATION_TOOLS}")

## Event handler helper

Prints events and captures the `run_context` + `tool_call_id` from any `HumanInputRequiredEvent`.

In [5]:
async def run_agent(agent, params, run_context=None):
    """Run agent, print events, return (run_context, tool_call_id or None).

    If the agent hits an ask_human interrupt, returns the tool_call_id needed for resume.
    If the agent completes, returns tool_call_id=None.
    """
    tool_call_id = None

    async for event in agent.astream(params, run_context=run_context):
        if isinstance(event, ToolCallingEvent):
            print(f"  TOOL_CALL: {event.data.tool_call.tool_name}")

        elif isinstance(event, ToolResultEvent):
            content = str(event.data.result.content)
            preview = content[:200] + "..." if len(content) > 200 else content
            print(f"  TOOL_RESULT: {preview}")

        elif isinstance(event, ThinkingEvent):
            print(f"  THINKING: {event.data.thinking_content}", end="")

        elif isinstance(event, StreamingTokenEvent):
            print(event.data.token, end="")

        elif isinstance(event, HumanInputRequiredEvent):
            tool_call_id = event.data.tool_call_id
            print(f"\n--- INTERRUPT: ask_human ---")
            print(f"  Question: {event.data.human_input.question}")
            if event.data.human_input.options:
                print(f"  Options: {event.data.human_input.options}")

        elif isinstance(event, HumanResponseEvent):
            print(f"  RESUMED with human response")

        elif isinstance(event, CompletedEvent):
            print(f"\n--- COMPLETED ---")

        run_context = event.run_context

    return run_context, tool_call_id

## Step 1: Initial query

The LLM calls `set_datetime_tool` → MCP returns result → `_check_tool_confirmation` fires → **execution halts automatically**.

In [6]:
rc, tid = await run_agent(
    agent,
    EIEAgentInputSchema(query="Show me NO2 air quality data for Houston TX from January to June 2023"),
)

print(f"\ntool_call_id to resume: {tid}")

  TOOL_CALL: ask_human

--- INTERRUPT: ask_human ---
  Question: To pull NO₂ air-quality data, I’ll need to (1) confirm the time range and (2) confirm the location. First: should I use the date range 2023-01-01/2023-06-30 for “January to June 2023”?
  Options: ['Yes, use 2023-01-01/2023-06-30', 'No, I meant a different range']

tool_call_id to resume: call_ez5HOoX57rSBf55Nc3rjlbhe


## Step 2: Confirm datetime → auto-interrupt after get_place

Resume with `HumanResponse`. The LLM calls `get_place_tool` → hook fires again → **halts**.

In [17]:
rc.human_response = HumanResponse(tool_call_id=tid, content="yes")

rc, tid = await run_agent(
    agent,
    EIEAgentInputSchema(query=""),  # params still required but ignored on resume
    run_context=rc,
)

print(f"\ntool_call_id to resume: {tid}")

  RESUMED with human response
  TOOL_CALL: ask_human

--- INTERRUPT: ask_human ---
  Question: Im missing the stored Houston geometry in this step. Please paste/provide the AOI as GeoJSON (Polygon/MultiPolygon), or tell me to re-geocode Houston again.
  Options: ['Re-geocode Houston', 'I will paste GeoJSON']

tool_call_id to resume: call_l3wC2aeTli3FmuFIlYt2WkV2


## Step 3: Confirm place → auto-interrupt after collections_rag

Resume → LLM calls `collections_rag_tool` → hook builds numbered list → **halts**.

In [ ]:
rc.human_response = HumanResponse(tool_call_id=tid, content="yes")

rc, tid = await run_agent(
    agent,
    EIEAgentInputSchema(query="yes"),
    run_context=rc,
)

print(f"\ntool_call_id to resume: {tid}")

## Step 4: Select collection → runs to completion

No more confirmation tools after this. The LLM runs `stac_search_tool` → `stats_tool` + `viz_tool` → **completes**.

In [ ]:
rc.human_response = HumanResponse(tool_call_id=tid, content="1")

rc, tid = await run_agent(
    agent,
    EIEAgentInputSchema(query="1"),
    run_context=rc,
)

# tid should be None here — no more interrupts, agent completed
print(f"\nCompleted: {tid is None}")
print(f"Total usage: {rc.usage}")

## How it works under the hood

```
User query
    │
    ▼
┌─────────────────────────────────────────────────────────────┐
│  Runner invocation 1                                        │
│  LLM call 1: → set_datetime_tool (MCP executes)            │
│  _check_tool_confirmation("set_datetime_tool") → interrupt  │
│  stream.cancel() ──────────────────────────────────────────│──► HumanInputRequiredEvent
└─────────────────────────────────────────────────────────────┘
    │
    user confirms "yes"  →  HumanResponse(tool_call_id, "yes")
    │
    ▼
┌─────────────────────────────────────────────────────────────┐
│  Runner invocation 2 (resumed)                              │
│  inject synthetic ask_human result into messages            │
│  LLM call 1: → get_place_tool (MCP executes)               │
│  _check_tool_confirmation("get_place_tool") → interrupt     │
│  stream.cancel() ──────────────────────────────────────────│──► HumanInputRequiredEvent
└─────────────────────────────────────────────────────────────┘
    ... (same for collections_rag_tool) ...
    │
    ▼
┌─────────────────────────────────────────────────────────────┐
│  Runner invocation 4 (final — no confirmation tools)        │
│  LLM call 1: → stac_search_tool                            │
│  LLM call 2: → stats_tool + viz_tool                       │
│  LLM call 3: → final text                                  │──► CompletedEvent
└─────────────────────────────────────────────────────────────┘
```

**Key:** The LLM never calls `ask_human` — it doesn't even know confirmations exist. The `_check_tool_confirmation` hook in `EIEAgent` checks the tool name after every MCP result and forces the interrupt in Python code. The LLM just sees a natural `ask_human → response` pair in its message history when it resumes.